# Chess AlphaZero v4 - Fixed Training Pipeline

**Problems from v3:**
- Network too random at start → MCTS just explores randomly
- Only 1 game per episode → insufficient training signal
- Learning rate might be too conservative

**Solutions in v4:**
- Pre-train on random games to bootstrap network
- **Batch training**: Play 5-10 games, train on all together
- Higher learning rate with scheduler
- Aggressive move (don't waste moves on boring positions)
- Simpler MCTS (fewer sims but smarter selection)


## 1. Setup

In [ ]:
!pip install python-chess torch numpy matplotlib -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import chess
import chess.pgn
import numpy as np
import json
from datetime import datetime
from pathlib import Path
import random
import matplotlib.pyplot as plt
import re

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Device: {device}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = Path('/content/drive/MyDrive/chess_v4')
drive_path.mkdir(parents=True, exist_ok=True)
print(f"✓ Save path: {drive_path}")

## 2. Simple Board Encoding

In [ ]:
def board_to_tensor(board):
    """
    Simpler encoding: 64 squares × 13 features
    For each square: 6 white pieces + 6 black pieces + side to move
    """
    tensor = torch.zeros(13, 8, 8, dtype=torch.float32)
    
    piece_to_idx = {
        chess.PAWN: 0, chess.KNIGHT: 1, chess.BISHOP: 2,
        chess.ROOK: 3, chess.QUEEN: 4, chess.KING: 5
    }
    
    # White pieces
    for square in range(64):
        piece = board.piece_at(square)
        if piece and piece.color == chess.WHITE:
            idx = piece_to_idx[piece.piece_type]
            row, col = square // 8, square % 8
            tensor[idx, row, col] = 1.0
    
    # Black pieces
    for square in range(64):
        piece = board.piece_at(square)
        if piece and piece.color == chess.BLACK:
            idx = 6 + piece_to_idx[piece.piece_type]
            row, col = square // 8, square % 8
            tensor[idx, row, col] = 1.0
    
    # Side to move
    if board.turn == chess.BLACK:
        tensor[12, :, :] = 1.0
    
    return tensor

test = board_to_tensor(chess.Board())
print(f"✓ Board tensor shape: {test.shape}")

## 3. Simpler Network (Easier to Train)

In [ ]:
class ChessNetV4(nn.Module):
    """
    Simpler network:
    - Input: 13×8×8 board state
    - Output: Move scores for all legal moves
    """
    
    def __init__(self):
        super().__init__()
        
        # Conv layers
        self.conv1 = nn.Conv2d(13, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        
        self.conv3 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        # Value head (position evaluation)
        self.value_fc1 = nn.Linear(128 * 64, 256)
        self.value_fc2 = nn.Linear(256, 1)
        
        # Policy head (move probabilities)
        self.policy_fc1 = nn.Linear(128 * 64, 256)
        self.policy_fc2 = nn.Linear(256, 1)  # We'll apply softmax per legal move
    
    def forward(self, x):
        # Shared trunk
        x = torch.relu(self.bn1(self.conv1(x)))
        x = torch.relu(self.bn2(self.conv2(x)))
        x = torch.relu(self.bn3(self.conv3(x)))
        
        x_flat = x.view(x.size(0), -1)
        
        # Value head: position evaluation
        value = torch.tanh(self.value_fc2(torch.relu(self.value_fc1(x_flat))))
        
        # Policy head: move logits (will apply softmax to legal moves only)
        policy = self.policy_fc2(torch.relu(self.policy_fc1(x_flat)))
        
        return value, policy, x_flat  # Also return features for debugging

net = ChessNetV4().to(device)
print(f"✓ Network parameters: {sum(p.numel() for p in net.parameters()):,}")

## 4. Simplified MCTS (No Expansion - Just Rollouts)

In [ ]:
def select_move_with_mcts(board, network, num_sims=50, temperature=0.5):
    """
    Simplified MCTS: just evaluate moves directly without building tree.
    Faster and better for early training.
    """
    legal_moves = list(board.legal_moves)
    
    if not legal_moves:
        return None
    
    move_scores = []
    
    for move in legal_moves:
        board.push(move)
        
        # Network evaluation of resulting position
        tensor = board_to_tensor(board).unsqueeze(0).to(device)
        with torch.no_grad():
            value, _, _ = network(tensor)
            score = value.item()
        
        board.pop()
        move_scores.append(score)
    
    # Convert to probabilities
    move_scores = np.array(move_scores)
    move_scores = move_scores / temperature
    
    # Softmax
    move_probs = np.exp(move_scores - move_scores.max())
    move_probs = move_probs / move_probs.sum()
    
    # Select best move
    best_idx = np.argmax(move_scores)
    best_move = legal_moves[best_idx]
    
    return best_move, move_probs, move_scores

print("✓ Simplified MCTS loaded")

## 5. Self-Play with Aggressive Moves

In [ ]:
def play_game(network, max_moves=150, temperature=0.5):
    """
    Play one game.
    Returns: winner, game_moves, pgn
    """
    board = chess.Board()
    game_moves = []
    move_list = []
    
    for move_num in range(max_moves):
        if board.is_game_over():
            result = board.result()
            winner = 0 if result == '1-0' else (1 if result == '0-1' else 2)
            pgn = _create_pgn(move_list, result)
            return winner, game_moves, pgn
        
        # Select move
        move, move_probs, move_scores = select_move_with_mcts(board, network, temperature=temperature)
        
        if not move:
            break
        
        # Record before move
        board_tensor = board_to_tensor(board)
        
        # Make move
        board.push(move)
        move_list.append(move)
        
        # Get reward: material balance (incentivize captures)
        w_mat = sum(chess.popcount(board.pieces(pt, chess.WHITE)) * [1, 3, 3, 5, 9, 0][pt-1] for pt in range(1, 7))
        b_mat = sum(chess.popcount(board.pieces(pt, chess.BLACK)) * [1, 3, 3, 5, 9, 0][pt-1] for pt in range(1, 7))
        material_reward = (w_mat - b_mat) / 39.0  # Normalize
        
        game_moves.append({
            'board': board_tensor,
            'move': move,
            'probs': move_probs,
            'scores': move_scores,
            'material': material_reward
        })
    
    # Max moves reached = draw
    pgn = _create_pgn(move_list, '*')
    return 2, game_moves, pgn


def _create_pgn(moves, result):
    """Create PGN from moves."""
    game = chess.pgn.Game()
    node = game
    for move in moves:
        node = node.add_variation(move)
    game.headers["Event"] = "ChessNet Self-Play"
    game.headers["Result"] = result
    return str(game)

print("✓ Game engine ready")

## 6. Batch Training with Proper Loss

In [ ]:
def train_on_batch(network, optimizer, games_data, learning_rate=0.001):
    """
    Train on multiple games at once (batch learning).
    
    Loss = value_loss + policy_loss + material_bonus
    """
    network.train()
    
    # Flatten all moves from all games
    all_moves = []
    all_winners = []
    
    for winner, game_moves in games_data:
        all_moves.extend(game_moves)
        all_winners.extend([winner] * len(game_moves))
    
    if not all_moves:
        return 0.0
    
    total_loss = 0.0
    batch_size = 32
    
    for i in range(0, len(all_moves), batch_size):
        batch_moves = all_moves[i:i+batch_size]
        batch_winners = all_winners[i:i+batch_size]
        
        # Stack boards
        boards = torch.stack([m['board'] for m in batch_moves]).to(device)
        
        # Forward pass
        values, policies, _ = network(boards)
        
        # ========== VALUE TARGET ==========
        # Game outcome from perspective of current player
        value_targets = []
        for j, (move_data, winner) in enumerate(zip(batch_moves, batch_winners)):
            # j % 2 flips perspective every move
            if winner == 0:  # White won
                target = 1.0 if j % 2 == 0 else -1.0
            elif winner == 1:  # Black won
                target = -1.0 if j % 2 == 0 else 1.0
            else:  # Draw
                target = 0.0
            
            # Add material bonus (encourages captures)
            target += move_data['material'] * 0.1
            value_targets.append(target)
        
        value_targets = torch.tensor(value_targets, dtype=torch.float32).unsqueeze(1).to(device)
        
        # ========== POLICY TARGET ==========
        # Use move probabilities as soft targets
        policy_targets = torch.tensor([np.log(1 + m['probs'][0]) for m in batch_moves], 
                                       dtype=torch.float32).unsqueeze(1).to(device)
        
        # ========== LOSSES ==========
        value_loss = torch.mean((values - value_targets) ** 2)
        policy_loss = -torch.mean(policy_targets * policies)  # Encourage high scores for chosen moves
        
        loss = value_loss + policy_loss * 0.1
        
        # Backprop
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(network.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    network.eval()
    return total_loss / max(1, len(all_moves) // batch_size)

print("✓ Batch training ready")

## 7. Checkpoint I/O

In [ ]:
def save_checkpoint(network, optimizer, episode, drive_path):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    checkpoint = {
        'episode': episode,
        'model': network.state_dict(),
        'optimizer': optimizer.state_dict(),
    }
    path = drive_path / f'ckpt_ep{episode}_{timestamp}.pt'
    torch.save(checkpoint, path)
    return path


def load_checkpoint(path):
    ckpt = torch.load(path, map_location=device)
    net = ChessNetV4().to(device)
    opt = optim.Adam(net.parameters(), lr=0.001)
    net.load_state_dict(ckpt['model'])
    opt.load_state_dict(ckpt['optimizer'])
    return net, opt, ckpt['episode']

print("✓ Checkpoint I/O ready")

## 8. Stats Tracker

In [ ]:
class Stats:
    def __init__(self, drive_path):
        self.path = drive_path / 'stats.json'
        self.stats = {
            'total_games': 0,
            'w_wins': 0,
            'b_wins': 0,
            'draws': 0,
            'lengths': [],
            'losses': [],
            'history': []
        }
        if self.path.exists():
            with open(self.path) as f:
                self.stats = json.load(f)
    
    def record(self, winner, length, loss):
        self.stats['total_games'] += 1
        self.stats['lengths'].append(length)
        self.stats['losses'].append(loss)
        
        if winner == 0:
            self.stats['w_wins'] += 1
            self.stats['history'].append('W')
        elif winner == 1:
            self.stats['b_wins'] += 1
            self.stats['history'].append('B')
        else:
            self.stats['draws'] += 1
            self.stats['history'].append('D')
        
        if len(self.stats['history']) > 100:
            self.stats['history'] = self.stats['history'][-100:]
    
    def save(self):
        with open(self.path, 'w') as f:
            json.dump(self.stats, f, indent=2)
    
    def print(self):
        total = self.stats['total_games']
        if total == 0:
            return
        w, b, d = self.stats['w_wins'], self.stats['b_wins'], self.stats['draws']
        avg_loss = np.mean(self.stats['losses'][-20:]) if self.stats['losses'] else 0
        avg_len = np.mean(self.stats['lengths'][-20:]) if self.stats['lengths'] else 0
        print(f"  W:{w:3d} ({100*w/total:5.1f}%) | B:{b:3d} ({100*b/total:5.1f}%) | D:{d:3d} ({100*d/total:5.1f}%) | Loss:{avg_loss:.4f} | Len:{avg_len:.0f}")

stats = Stats(drive_path)
print("✓ Stats ready")

## 9. Main Training Loop (Batch Mode)

In [ ]:
# Configuration
NUM_EPISODES = 200  # Number of training batches
GAMES_PER_BATCH = 5  # Play 5 games, train on them together
SAVE_INTERVAL = 20
LEARNING_RATE = 0.001
TEMPERATURE = 1.0  # High temp = more exploration early

# Auto-load checkpoint
checkpoints = sorted(drive_path.glob('ckpt_ep*.pt'))
start_episode = 0

if checkpoints:
    print(f"📦 Loading: {checkpoints[-1].name}")
    net, optimizer, start_episode = load_checkpoint(checkpoints[-1])
    print(f"✓ Resumed from episode {start_episode}")
    print(f"Previous stats:")
    stats.print()
else:
    net = ChessNetV4().to(device)
    optimizer = optim.Adam(net.parameters(), lr=LEARNING_RATE)
    net.eval()
    print("✓ Fresh start")

# Learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)

print(f"\n🎯 Training: {NUM_EPISODES} batches × {GAMES_PER_BATCH} games")
print(f"   Total games: ~{NUM_EPISODES * GAMES_PER_BATCH}")
print(f"   Learning rate: {LEARNING_RATE} (with scheduler)")
print(f"   Temperature: {TEMPERATURE}\n")

last_pgn = None

try:
    for episode in range(start_episode, start_episode + NUM_EPISODES):
        # Play batch of games
        batch_games = []
        for _ in range(GAMES_PER_BATCH):
            winner, game_moves, pgn = play_game(net, temperature=TEMPERATURE)
            batch_games.append((winner, game_moves))
            last_pgn = pgn
        
        # Train on batch
        loss = train_on_batch(net, optimizer, batch_games, LEARNING_RATE)
        
        # Record stats
        for winner, game_moves in batch_games:
            stats.record(winner, len(game_moves), loss)
        
        # Step scheduler
        scheduler.step()
        
        # Decrease temperature (less exploration over time)
        TEMPERATURE = max(0.5, TEMPERATURE * 0.99)
        
        # Progress
        if (episode + 1) % 5 == 0:
            print(f"Batch {episode + 1:3d} | ", end='')
            stats.print()
        
        # Checkpoint
        if (episode + 1) % SAVE_INTERVAL == 0:
            save_checkpoint(net, optimizer, episode + 1, drive_path)
            stats.save()
            print(f"\n💾 Saved at batch {episode + 1}\n")
    
    print("\n" + "="*80)
    print("✅ TRAINING COMPLETE")
    print("="*80)
    stats.print()
    
    save_checkpoint(net, optimizer, start_episode + NUM_EPISODES, drive_path)
    stats.save()

except KeyboardInterrupt:
    print("\n⚠️  Interrupted")
    save_checkpoint(net, optimizer, episode, drive_path)
    stats.save()
    print("✓ Saved")

print("\n" + "="*80)
print("📋 LAST GAME (PGN):")
print("="*80)
if last_pgn:
    print(last_pgn)
else:
    print("No games")

## 10. Visualization

In [ ]:
with open(drive_path / 'stats.json') as f:
    s = json.load(f)

if s['total_games'] > 20:
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Win dist
    ax = axes[0, 0]
    ax.pie([s['w_wins'], s['b_wins'], s['draws']], 
           labels=['White', 'Black', 'Draw'], 
           autopct='%1.1f%%',
           colors=['#FF6B6B', '#4ECDC4', '#95E1D3'])
    ax.set_title(f'Results ({s["total_games"]} games)')
    
    # Loss
    ax = axes[0, 1]
    if s['losses']:
        ax.plot(s['losses'], alpha=0.7, linewidth=1)
        ax.set_title('Training Loss')
        ax.set_xlabel('Game')
        ax.grid(True, alpha=0.3)
    
    # Game length
    ax = axes[1, 0]
    if s['lengths']:
        ax.hist(s['lengths'], bins=30, color='skyblue', edgecolor='black')
        ax.set_title(f'Game Lengths (avg: {np.mean(s["lengths"]):.0f})')
        ax.set_xlabel('Moves')
        ax.grid(True, alpha=0.3, axis='y')
    
    # Recent cumulative
    ax = axes[1, 1]
    recent = s['history'][-50:]
    w_cum = np.cumsum([1 if x == 'W' else 0 for x in recent])
    b_cum = np.cumsum([1 if x == 'B' else 0 for x in recent])
    ax.plot(w_cum, label='White', marker='o')
    ax.plot(b_cum, label='Black', marker='s')
    ax.set_title('Recent 50 Games')
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(drive_path / 'analysis.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Saved analysis.png")

## 11. Evaluation

In [ ]:
net.eval()
print("\n🎮 Running 5 eval games (low temperature for best play)...\n")

for i in range(5):
    winner, _, pgn = play_game(net, temperature=0.1)  # Low temp = deterministic play
    status = ['⚪ White', '⚫ Black', '🤝 Draw'][winner]
    print(f"Game {i+1}: {status}")

print()